# Supervised Tabular Training

This notebook trains a compact flat classifier on the bundled Wine dataset. It follows the same API as Hello World, but uses a few more numeric inputs and exposes root embeddings during the same run. Use it as a tabular comparison point, not as the main nested-data story.

Import the runtime pieces used in the full training loop: Lightning for optimization and Polars for reading the bundled JSONL records.


In [1]:
import lightning.pytorch as lit
import polars as pl
import torch
from loguru import logger
from rich.pretty import pprint

import json2vec as j2v

logger.remove()


The Wine buffer is flat, but it gives enough numeric variation to make a clearer supervised example than handmade records. The model will use four chemistry fields to predict the cultivar.


In [2]:
records = pl.read_ndjson("docs/data/wine.jsonl").head(48)

records.head()


alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280_od315_of_diluted_wines,proline,cultivar
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
14.23,1.71,2.43,15.6,127.0,2.8,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,"""class_0"""
12.37,0.94,1.36,10.6,88.0,1.98,0.57,0.28,0.42,1.95,1.05,1.82,520.0,"""class_1"""
12.86,1.35,2.32,18.0,122.0,1.51,1.25,0.21,0.94,4.1,0.76,1.29,630.0,"""class_2"""
13.2,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.4,1050.0,"""class_0"""
12.33,1.1,2.28,16.0,101.0,2.05,1.09,0.63,0.41,3.27,1.25,1.67,680.0,"""class_1"""


The schema is the architecture. Four `Number` requests feed the root encoder, `cultivar` is the categorical target, and `embed=True` asks the root node to return embeddings after training.


In [3]:
model = j2v.Model.from_schema(
    j2v.Number("alcohol"),
    j2v.Number("malic_acid"),
    j2v.Number("color_intensity"),
    j2v.Number("proline"),
    j2v.Category("cultivar", target=True, max_vocab_size=4, topk=[2]),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    embed=True,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)


`PolarsDataModule(...)` reads the schema configuration from the model, so batch size, queries, targets, and tensorfield behavior stay tied to one object.

In [4]:
datamodule = j2v.PolarsDataModule(
    model,
    train=records,
    validate=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=32,
    chunk_batch_size=32,
    sample_rate=1.0,
)

The tutorial trains for one tiny pass. In a real experiment this is where you would scale epochs, validation splits, callbacks, logging, and checkpointing.

In [5]:
trainer = lit.Trainer(
    accelerator="cpu",
    max_epochs=1,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    enable_checkpointing=False,
    limit_train_batches=1,
    limit_val_batches=1,
)

trainer.fit(model=model, datamodule=datamodule)

GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer(limit_train_batches=1)` was configured so 1 batch per epoch will be used.


`Trainer(limit_val_batches=1)` was configured so 1 batch will be used.


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
`Trainer.fit` stopped: `max_epochs=1` reached.


The plot confirms that the target and root embedding are both configured before inference.

In [6]:
model.plot()

Schema
record [array] embed
record
d_model=16  attention=mha  max_length=1  n_outputs=1  n_layers=1  n_heads=4  batch_size=8  parameters=26,039  arrays=1  
fields=5  targets=1  embeds=1  embed=True  n_linear=1
├── alcohol [number] 3,943 params
│   record/alcohol
│   query=[*].alcohol  pooling=query  objective=mae  weight=1.0  n_heads=4  n_linear=1  jitter=0.0  n_bands=8  offset=4
├── malic_acid [number] 3,943 params
│   record/malic_acid
│   query=[*].malic_acid  pooling=query  objective=mae  weight=1.0  n_heads=4  n_linear=1  jitter=0.0  n_bands=8  
│   offset=4
├── color_intensity [number] 3,943 params
│   record/color_intensity
│   query=[*].color_intensity  pooling=query  objective=mae  weight=1.0  n_heads=4  n_linear=1  jitter=0.0  n_bands=8  
│   offset=4
├── proline [number] 3,943 params
│   record/proline
│   query=[*].proline  pooling=query  objective=mae  weight=1.0  n_heads=4  n_linear=1  jitter=0.0  n_bands=8  offset=4
└── cultivar [category] target 3,659 params
    record/

After training, `predict` returns typed outputs for supervised targets and `embed` returns vectors from the nodes configured with `embed=True`.

In [7]:
batch = [[record] for record in records.to_dicts()[:3]]
pprint(model.predict(batch))
pprint(model.embed(batch))

{
│   'record/cultivar': {
│   │   'state': {
│   │   │   'valued': [0.5733041763305664, 0.5707119703292847, 0.5714324116706848],
│   │   │   'null': [0.0697474330663681, 0.07054644078016281, 0.06922597438097],
│   │   │   'padded': [0.043713685125112534, 0.04403073713183403, 0.0426005944609642],
│   │   │   'masked': [0.21819572150707245, 0.21992768347263336, 0.224310502409935],
│   │   │   'other': [0.09503911435604095, 0.09478320926427841, 0.09243053942918777]
│   │   },
│   │   'content': {
│   │   │   'value': ['class_0', 'class_0', 'class_0'],
│   │   │   'probability': [0.4861387312412262, 0.4873562753200531, 0.4873278737068176],
│   │   │   'topk': [
│   │   │   │   [
│   │   │   │   │   {'label': 'class_0', 'probability': 0.4861387312412262},
│   │   │   │   │   {'label': 'class_2', 'probability': 0.29188817739486694}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'class_0', 'probability': 0.4873562753200531},
│   │   │   │   │   {'label': 'class_2', 'probability': 0.28949180245399475}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'class_0', 'probability': 0.4873278737068176},
│   │   │   │   │   {'label': 'class_2', 'probability': 0.29495975375175476}
│   │   │   │   ]
│   │   │   ]
│   │   }
│   }
}

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   -0.3165125548839569,
│   │   │   │   0.3732002079486847,
│   │   │   │   -0.32456210255622864,
│   │   │   │   -0.2500506639480591,
│   │   │   │   -0.06044051796197891,
│   │   │   │   0.13174597918987274,
│   │   │   │   -0.09010523557662964,
│   │   │   │   -0.37949612736701965,
│   │   │   │   0.15350522100925446,
│   │   │   │   0.34721818566322327,
│   │   │   │   0.0861932635307312,
│   │   │   │   -0.26976945996284485,
│   │   │   │   0.3152990937232971,
│   │   │   │   0.25633370876312256,
│   │   │   │   -0.12248498946428299,
│   │   │   │   0.12278846651315689
│   │   │   ],
│   │   │   [
│   │   │   │   -0.3239968419075012,
│   │   │   │   0.38756880164146423,
│   │   │   │   -0.33263054490089417,
│   │   │   │   -0.2460600584745407,
│   │   │   │   -0.0671217292547226,
│   │   │   │   0.13607507944107056,
│   │   │   │   -0.08765871077775955,
│   │   │   │   -0.36148035526275635,
│   │   │   │   0.14231759309768677,
│   │   │   │   0.3405202627182007,
│   │   │   │   0.10055132210254669,
│   │   │   │   -0.277523010969162,
│   │   │   │   0.3120068907737732,
│   │   │   │   0.25534701347351074,
│   │   │   │   -0.11727564036846161,
│   │   │   │   0.11230329424142838
│   │   │   ],
│   │   │   [
│   │   │   │   -0.3099939525127411,
│   │   │   │   0.35480281710624695,
│   │   │   │   -0.3266640305519104,
│   │   │   │   -0.2518652677536011,
│   │   │   │   -0.043875422328710556,
│   │   │   │   0.1575743556022644,
│   │   │   │   -0.0996171161532402,
│   │   │   │   -0.37931323051452637,
│   │   │   │   0.17773237824440002,
│   │   │   │   0.3415555953979492,
│   │   │   │   0.04578641057014465,
│   │   │   │   -0.2914011478424072,
│   │   │   │   0.3309769034385681,
│   │   │   │   0.23798620700836182,
│   │   │   │   -0.09889163821935654,
│   │   │   │   0.12686049938201904
│   │   │   ]
│   │   ]
│   }
}